# Qualidade de `usuarios.csv`

Diagnóstico do cadastro de contas pelos mesmos cinco critérios usados nos outros arquivos:

| Critério | Pergunta |
|---|---|
| **Completude** | Está tudo lá? Campos preenchidos e contas que deveriam existir |
| **Validade** | Os valores estão no formato e no domínio esperados? |
| **Consistência** | Os dados concordam entre si, dentro do arquivo e com os outros arquivos? |
| **Unicidade** | Cada conta (e cada pessoa) aparece uma vez só? |
| **Temporalidade** | As datas estão no período certo, sem buracos e sem mudança de universo? |

**Só diagnóstico:** nada é corrigido aqui. Cada achado vai para uma tabela-resumo em `saidas/`, com status
(`ok`, `esperado` ou `atenção`) e tratamento sugerido. Nada em `dados/brutos/` é alterado (hash conferido no início e no fim).

**Decisões já registradas que valem aqui** (`decisoes.md`): contas de teste (e-mail `@vitaliza.com.br`) são removidas;
datas dos logs convertidas para o horário de Brasília.

**Privacidade:** e-mail e cidade (texto livre) aparecem só como contagem e padrão, nunca ao lado do `user_id`.

In [1]:
import hashlib
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BRUTOS = ROOT / "dados" / "brutos"
SAIDAS = ROOT / "saidas"
LANCAMENTO = pd.Timestamp("2020-09-01")   # lançamento do app (dicionário e enunciado)
INICIO_LOG = pd.Timestamp("2024-07-01")   # início de eventos_assinatura.csv e pagamentos.csv
CORTE = pd.Timestamp("2026-09-30")
FIM_CORTE = CORTE + pd.Timedelta(days=1)  # 01/10/2026 00:00, horário de Brasília
TZ_BR = "America/Sao_Paulo"

ARQUIVOS = ["usuarios.csv", "assinaturas.csv", "eventos_assinatura.csv", "pagamentos.csv", "uso_mensal.csv",
            "atividades.parquet", "amizades.csv", "campanha_ago26.csv", "onboarding.csv", "pesquisa_saida.csv",
            "feed_posts.csv", "tickets_suporte.csv"]

def sha1(path):
    return hashlib.sha1(path.read_bytes()).hexdigest()

hash_inicio = {f: sha1(BRUTOS / f) for f in ARQUIVOS}

achados = []

def achado(criterio, checagem, resultado, linhas, tratamento, status=None):
    # status: ok (nada encontrado), esperado (encontrado, mas é comportamento do negócio) ou atenção.
    if status is None:
        status = "ok" if linhas == 0 else "atenção"
    achados.append({"criterio": criterio, "checagem": checagem, "status": status,
                    "resultado": resultado, "linhas_afetadas": int(linhas), "tratamento_sugerido": tratamento})

def ler(nome, **kwargs):
    # Lê como gravado: tudo texto, vazio como "" (parquet mantém os tipos do arquivo).
    if nome.endswith(".parquet"):
        return pd.read_parquet(BRUTOS / nome, **kwargs)
    return pd.read_csv(BRUTOS / nome, dtype=str, keep_default_na=False, **kwargs)

def para_brasilia(texto):
    # Data e hora terminadas em Z estão em UTC; as demais, em horário de Brasília. Devolve Brasília sem fuso.
    eh_utc = texto.str.endswith("Z")
    utc = pd.Series(pd.NaT, index=texto.index, dtype="datetime64[ns, UTC]")
    utc[eh_utc] = pd.to_datetime(texto[eh_utc], format="%Y-%m-%dT%H:%M:%SZ", errors="coerce", utc=True)
    utc[~eh_utc] = (pd.to_datetime(texto[~eh_utc], format="%Y-%m-%d %H:%M:%S", errors="coerce")
                    .dt.tz_localize(TZ_BR).dt.tz_convert("UTC"))
    return utc.dt.tz_convert(TZ_BR).dt.tz_localize(None)

## 0. Leitura

Tudo como texto e sem converter vazio em nulo, para ver o arquivo exatamente como chegou. Dos outros arquivos, só as
colunas que as checagens pedem.

In [2]:
usuarios = ler("usuarios.csv")
assert len(usuarios) == 64_948  # linhas segundo o dicionário (seção 02)
N = len(usuarios)
print(f"usuarios: {N:,} linhas x {usuarios.shape[1]} colunas -> {list(usuarios.columns)}")
usuarios.drop(columns=["email", "cidade"]).sample(5, random_state=0)

usuarios: 64,948 linhas x 12 colunas -> ['user_id', 'email', 'data_cadastro', 'data_nascimento', 'sexo', 'cidade', 'uf', 'cep_prefixo', 'canal_aquisicao', 'sistema', 'versao_app', 'relogio_conectado']


,user_id,data_cadastro,data_nascimento,sexo,uf,cep_prefixo,canal_aquisicao,sistema,versao_app,relogio_conectado
47685,u_e37ae3b4,2026-09-14 21:17:16,1989-09-21,,SP,055,google_ads,android,5.4.0,nenhum
29287,u_b1e2c06c,2026-04-08 06:04:42,1987-10-01,F,SP,042,influenciador,ios,5.4.1,nenhum
31149,u_e760ce8f,2026-10-21 17:45:50,1969-03-21,feminino,RJ,204,organico,android,4.9.8,nenhum
6048,u_7c6ecaf6,2026-07-17 07:26:52,1983-06-17,F,SC,880,organico,android,5.3.2,nenhum
59582,u_e2263926,2024-10-01 21:10:53,1978-09-21,Fem,PR,819,organico,android,5.4.1,nenhum


### Colunas derivadas para a análise

- `dc`, `data_cad`, `mes_cad`, `ano_cad`: data de cadastro convertida (o texto original não muda);
- `email_norm`: e-mail em minúsculas e sem espaços nas pontas; `eh_teste`: termina em `@vitaliza.com.br`;
- `nasc`, `idade`: nascimento convertido e idade no corte;
- `sexo_norm`, `sistema_norm`, `cidade_norm`, `cep3`: as grafias propostas na seção 2, usadas só para medir o problema.

In [3]:
u = usuarios.copy()
u["dc"] = pd.to_datetime(u["data_cadastro"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
u["data_cad"] = u["dc"].dt.normalize()
u["mes_cad"] = u["dc"].dt.to_period("M")
u["ano_cad"] = u["dc"].dt.year
u["email_norm"] = u["email"].str.strip().str.lower()
u["eh_teste"] = u["email_norm"].str.endswith("@vitaliza.com.br")
u["nasc"] = pd.to_datetime(u["data_nascimento"].replace("", None), format="%Y-%m-%d", errors="coerce")
u["idade"] = (CORTE - u["nasc"]).dt.days / 365.25

SEXO = {"F": "F", "Fem": "F", "feminino": "F", "M": "M", "Masc": "M", "masculino": "M", "": ""}
CIDADE = {
    "São Paulo": ["São Paulo", "Sampa", "são paulo", "São Paulo - SP", "Sao Paulo", "S. Paulo", "SAO PAULO", "SP"],
    "Rio de Janeiro": ["Rio de Janeiro", "Rio de janeiro", "rio de janeiro", "RIO DE JANEIRO", "RJ", "Rio"],
    "Belo Horizonte": ["Belo Horizonte", "Belo horizonte", "B. Horizonte", "BELO HORIZONTE", "BH"],
    "Florianópolis": ["Florianópolis", "floripa", "Florianopolis", "Floripa"],
    "Brasília": ["Brasília", "Brasília - DF", "BSB", "brasilia", "Brasilia"],
}
MAPA_CIDADE = {grafia: nome for nome, grafias in CIDADE.items() for grafia in grafias}
u["sexo_norm"] = u["sexo"].map(SEXO)
u["sistema_norm"] = u["sistema"].str.strip().str.lower()
u["cidade_norm"] = u["cidade"].map(MAPA_CIDADE).fillna(u["cidade"])
u["cep3"] = u["cep_prefixo"].where(u["cep_prefixo"] == "", u["cep_prefixo"].str.zfill(3))
print("datas de cadastro que não converteram:", u["dc"].isna().sum())

datas de cadastro que não converteram: 0


## 1. Completude

Espera-se o `user_id`, o e-mail, a data de cadastro e as colunas preenchidas pelo app sem vazio. As colunas digitadas pela
pessoa (nascimento, sexo, CEP) "não passam por validação" (dicionário), então algum vazio é natural; o que importa é se a
taxa é estável ou se há vazio disfarçado.

In [4]:
VAZIOS_DISFARCADOS = {"nan", "null", "none", "na", "n/a", "-"}
colunas = list(usuarios.columns)
vazios = pd.DataFrame({
    "vazio": [(usuarios[c].str.strip() == "").sum() for c in colunas],
    "vazio_disfarcado": [usuarios[c].str.strip().str.lower().isin(VAZIOS_DISFARCADOS).sum() for c in colunas],
}, index=colunas)
vazios["% vazio"] = (100 * vazios["vazio"] / N).round(1)
vazios

,vazio,vazio_disfarcado,% vazio
user_id,0,0,0.0
email,0,0,0.0
data_cadastro,0,0,0.0
data_nascimento,3301,0,5.1
sexo,3142,0,4.8
cidade,0,0,0.0
uf,0,0,0.0
cep_prefixo,1894,0,2.9
canal_aquisicao,0,0,0.0
sistema,0,0,0.0


In [5]:
obrigatorias = ["user_id", "email", "data_cadastro", "cidade", "uf", "canal_aquisicao", "sistema", "versao_app",
                "relogio_conectado"]
n_obrig = int(vazios.loc[obrigatorias, ["vazio", "vazio_disfarcado"]].sum().sum())
achado("Completude", "vazio nas colunas preenchidas pelo app ou pelo marketing", f"{n_obrig:,}", n_obrig,
       "Nenhum")

### Vazio por ano de cadastro

Se a taxa de vazio mudasse de um ano para outro, seria sinal de mudança no formulário. Taxa estável indica campo
que parte das pessoas simplesmente não preenche.

In [6]:
taxa = pd.DataFrame({
    "nascimento vazio %": (u["data_nascimento"] == "").groupby(u["ano_cad"]).mean(),
    "sexo vazio %": (u["sexo"] == "").groupby(u["ano_cad"]).mean(),
    "CEP vazio %": (u["cep_prefixo"] == "").groupby(u["ano_cad"]).mean(),
    "contas": u.groupby("ano_cad").size(),
})
taxa.iloc[:, :3] = (100 * taxa.iloc[:, :3]).round(1)
display(taxa)

ate_corte = u["dc"] < FIM_CORTE
for col, nome in [("data_nascimento", "nascimento"), ("sexo", "sexo"), ("cep_prefixo", "CEP")]:
    t = taxa.loc[taxa.index <= 2026, f"{nome} vazio %"]
    n = int((u[col] == "").sum())
    achado("Completude", f"{nome} vazio", f"{n:,} ({100 * n / N:.1f}%); taxa por ano de {t.min()}% a {t.max()}%",
           n, "Campo digitado e opcional na prática: manter como 'não informado', sem imputar", status="esperado")

,nascimento vazio %,sexo vazio %,CEP vazio %,contas
ano_cad,,,,
2020,4.8,4.1,3.0,439
2021,5.4,4.4,3.1,1383
2022,4.8,4.8,2.8,1336
2023,6.2,4.8,3.3,1441
2024,5.1,4.8,2.8,11939
2025,5.1,4.8,2.8,26167
2026,5.0,4.9,3.1,22202
2027,7.3,7.3,2.4,41


### Vazio disfarçado em `data_nascimento`

Uma única data muito repetida, longe do resto, é o valor-padrão de um formulário e não um nascimento.

In [7]:
mais_repetidas = u["data_nascimento"].value_counts().head(5)
display(mais_repetidas.to_frame("contas"))
n_1900 = int((u["data_nascimento"] == "1900-01-01").sum())
segunda = u.loc[~u["data_nascimento"].isin(["", "1900-01-01", "2031-05-12"]), "data_nascimento"].value_counts().iloc[0]
print(f"1900-01-01: {n_1900:,} contas; fora 1900-01-01 e 2031-05-12, a data mais repetida tem {segunda:,}")
achado("Completude", "nascimento disfarçado de vazio (1900-01-01)",
       f"{n_1900:,} contas com 1900-01-01 (idade de 126 anos); nenhum outro ano antes de 1952", n_1900,
       "Tratar como nascimento não informado (somar aos vazios)")

,contas
data_nascimento,
,3301
1900-01-01,412
2031-05-12,194
1983-11-08,18
1978-04-13,17


1900-01-01: 412 contas; fora 1900-01-01 e 2031-05-12, a data mais repetida tem 18


In [8]:
n_teste = int(u["eh_teste"].sum())
interno = u["canal_aquisicao"] == "interno"
print(f"e-mail @vitaliza.com.br: {n_teste} | canal interno: {interno.sum()} | os dois: {(u['eh_teste'] & interno).sum()}")
achado("Completude", "contas de teste (@vitaliza.com.br)",
       f"{n_teste} contas, as mesmas 60 do canal 'interno'", n_teste,
       "Já registrado no decisoes.md: remover", status="atenção")

e-mail @vitaliza.com.br: 60 | canal interno: 60 | os dois: 60


### Quem está nos outros arquivos e não tem cadastro

Este é o arquivo que liga todos os outros: todo `user_id` do data room deveria estar aqui.

In [9]:
ids = set(u["user_id"])
outros = {
    "assinaturas.csv": ["user_id"], "eventos_assinatura.csv": ["user_id"], "pagamentos.csv": ["user_id"],
    "uso_mensal.csv": ["user_id"], "atividades.parquet": ["user_id"], "amizades.csv": ["seguidor_id", "seguido_id"],
    "campanha_ago26.csv": ["user_id"], "onboarding.csv": ["user_id"], "pesquisa_saida.csv": ["user_id"],
    "feed_posts.csv": ["user_id"], "tickets_suporte.csv": ["user_id"],
}
ids_por_arquivo, linhas_fora = {}, []
for arq, cols in outros.items():
    df = ler(arq, columns=cols) if arq.endswith(".parquet") else ler(arq, usecols=cols)
    for c in cols:
        s = df[c][df[c] != ""]
        ids_por_arquivo[f"{arq}:{c}"] = set(s)
        linhas_fora.append({"arquivo": arq, "coluna": c, "linhas": len(s), "contas": s.nunique(),
                            "linhas sem cadastro": int((~s.isin(ids)).sum())})
linhas_fora = pd.DataFrame(linhas_fora)
display(linhas_fora)
n_fora = int(linhas_fora["linhas sem cadastro"].sum())
achado("Completude", "user_id dos outros 11 arquivos sem cadastro aqui", f"{n_fora:,} linhas", n_fora, "Nenhum")

,arquivo,coluna,linhas,contas,linhas sem cadastro
0,assinaturas.csv,user_id,32643,32643,0
1,eventos_assinatura.csv,user_id,275565,52196,0
2,pagamentos.csv,user_id,165548,31782,0
3,uso_mensal.csv,user_id,591662,63830,0
4,atividades.parquet,user_id,1254475,36551,0
5,amizades.csv,seguidor_id,113480,50455,0
6,amizades.csv,seguido_id,113480,50547,0
7,campanha_ago26.csv,user_id,14120,14120,0
8,onboarding.csv,user_id,63929,63929,0
9,pesquisa_saida.csv,user_id,643,641,0


### Contas que deveriam existir: o cadastro antes de 07/2024

O dicionário diz que o arquivo traz "todas as contas desde o lançamento, em setembro de 2020, e a maior parte delas usa
só a versão grátis". Se for verdade, a proporção de contas que viram assinantes deve ser parecida em todas as épocas.
Testo a proporção por ano de cadastro, separando o antes e o depois de 07/2024 (mês em que começam os logs de eventos e
de pagamentos).

In [10]:
assinaturas = ler("assinaturas.csv", usecols=["user_id", "inicio", "plano_atual", "canal_cobranca", "status_atual"])
assinaturas["ini"] = pd.to_datetime(assinaturas["inicio"], format="%Y-%m-%d")
primeira = assinaturas.groupby("user_id")["ini"].min()
u["primeira_assinatura"] = u["user_id"].map(primeira)
u["assina"] = u["primeira_assinatura"].notna()
u["epoca"] = (u["dc"] >= INICIO_LOG).map({False: "antes de 07/2024", True: "07/2024 em diante"})

conv = u[u["dc"] < FIM_CORTE].groupby(["epoca", "ano_cad"]).agg(contas=("user_id", "size"), viraram_assinantes=("assina", "sum"))
conv["% assinantes"] = (100 * conv["viraram_assinantes"] / conv["contas"]).round(1)
display(conv)
por_epoca = u[u["dc"] < FIM_CORTE].groupby("epoca")["assina"].agg(["size", "mean"])
display(por_epoca.assign(mean=lambda d: (100 * d["mean"]).round(1)).rename(columns={"size": "contas", "mean": "% assinantes"}))

contas  viraram_assinantes  % assinantes
epoca             ano_cad                                          
07/2024 em diante 2024      11263                5317          47.2
                  2025      26167               12271          46.9
                  2026      22124                9848          44.5
antes de 07/2024  2020        439                 436          99.3
                  2021       1383                1360          98.3
                  2022       1336                1316          98.5
                  2023       1441                1425          98.9
                  2024        676                 670          99.1

,contas,% assinantes
epoca,,
07/2024 em diante,59554,46.1
antes de 07/2024,5275,98.7


O enunciado traz uma segunda prova: "cerca de 400 assinantes no fim de 2020". Se só ~440 contas foram criadas em 2020,
quase todas pagavam no fim do ano, o que não combina com um app freemium em que a maior parte usa só o grátis.

In [11]:
cad_2020 = int((u["ano_cad"] == 2020).sum())
ass_2020 = int(((u["ano_cad"] == 2020) & u["assina"]).sum())
print(f"contas cadastradas em 2020: {cad_2020}; delas, viraram assinantes: {ass_2020}; enunciado: ~400 assinantes no fim de 2020")
antes = u["dc"] < INICIO_LOG
n_antes = int(antes.sum())
p_antes = 100 * u.loc[antes, "assina"].mean()
p_depois = 100 * u.loc[~antes & (u["dc"] < FIM_CORTE), "assina"].mean()
achado("Completude", "contas grátis cadastradas antes de 07/2024 ausentes",
       f"{n_antes:,} contas antes de 07/2024, {p_antes:.1f}% viraram assinantes (contra {p_depois:.1f}% depois); "
       f"2020: {ass_2020} de {cad_2020}. Hipótese a confirmar com o dono do dado: na migração de 07/2024 só as contas "
       "que tinham assinado vieram", n_antes,
       "Não usar coortes de cadastro anteriores a 07/2024 para taxa de conversão, funil grátis -> pago ou perfil de quem "
       "não assina; confirmar com o dono do dado")

contas cadastradas em 2020: 439; delas, viraram assinantes: 436; enunciado: ~400 assinantes no fim de 2020


## 2. Validade

Cada categoria contra a lista observada (o dicionário não traz listas de valores), cada data convertendo e cada número
dentro de limites plausíveis.

In [12]:
padrao_id = u["user_id"].str.fullmatch(r"u_[0-9a-f]{8}")
achado("Validade", "user_id fora do padrão u_[0-9a-f]{8}", f"{(~padrao_id).sum():,}", int((~padrao_id).sum()), "Nenhum")

formato_email = u["email_norm"].str.fullmatch(r"[^@\s]+@[^@\s]+\.[a-z]{2,}")
achado("Validade", "e-mail com formato inválido", f"{(~formato_email).sum():,}", int((~formato_email).sum()), "Nenhum")
dominios = u["email_norm"].str.split("@").str[1].value_counts()
print("domínios (contagem):", dominios.to_dict())
maiusc = u["email"] != u["email_norm"]
print(f"e-mails com letra maiúscula: {maiusc.sum():,} (investigados na seção 4)")

falha_dc = int(u["dc"].isna().sum())
falha_nasc = int((u["nasc"].isna() & (u["data_nascimento"] != "")).sum())
padroes = u["data_cadastro"].str.replace(r"\d", "9", regex=True).value_counts().to_dict()
achado("Validade", "datas que não convertem (cadastro e nascimento)",
       f"cadastro: {falha_dc}; nascimento: {falha_nasc}; padrões do cadastro: {padroes}", falha_dc + falha_nasc, "Nenhum")

domínios (contagem): {'gmail.com': 21667, 'hotmail.com': 7308, 'yahoo.com.br': 7230, 'uol.com.br': 7193, 'icloud.com': 7187, 'outlook.com': 7177, 'bol.com.br': 7126, 'vitaliza.com.br': 60}
e-mails com letra maiúscula: 959 (investigados na seção 4)


### Categorias

Lista observada de cada coluna categórica. `sexo`, `sistema` e `cidade` têm grafias diferentes para o mesmo valor;
as demais vêm padronizadas.

In [13]:
for c in ["sexo", "sistema", "uf", "canal_aquisicao", "relogio_conectado", "versao_app"]:
    print(f"{c}: {u[c].value_counts().to_dict()}")

sexo: {'F': 19330, 'M': 17646, 'Fem': 6657, 'feminino': 6520, 'Masc': 5838, 'masculino': 5815, '': 3142}
sistema: {'android': 28027, 'ios': 20854, 'iOS': 7463, 'Android': 7124, 'IOS': 1480}
uf: {'SP': 31994, 'RJ': 7513, 'MG': 4550, 'PR': 3922, 'DF': 3268, 'RS': 3267, 'SC': 2674, 'GO': 1950, 'PE': 1940, 'CE': 1939, 'BA': 1931}
canal_aquisicao: {'organico': 18531, 'influenciador': 14067, 'meta_ads': 11939, 'indicacao': 8117, 'google_ads': 6801, 'parceria_corporativa': 5433, 'interno': 60}
relogio_conectado: {'nenhum': 40159, 'garmin': 13403, 'apple_watch': 11386}
versao_app: {'5.4.1': 20010, '5.4.0': 19996, '5.3.2': 9899, '5.3.0': 9795, '5.1.0': 2644, '4.9.8': 2604}


In [14]:
fora_sexo = u["sexo_norm"].isna().sum()
n_sexo = int((~u["sexo"].isin(["F", "M", ""])).sum())
achado("Validade", "sexo com grafias diferentes",
       f"6 grafias para F e M (F, Fem, feminino; M, Masc, masculino); {n_sexo:,} fora de F/M; "
       f"{fora_sexo} sem mapeamento", n_sexo, "Padronizar: F, Fem, feminino -> F; M, Masc, masculino -> M; vazio -> não informado")

n_sist = int((u["sistema"] != u["sistema_norm"]).sum())
achado("Validade", "sistema com grafias diferentes",
       f"5 grafias para 2 valores; {n_sist:,} fora de android/ios", n_sist, "Padronizar em minúsculas: android, ios")

n_cid = int((u["cidade"] != u["cidade_norm"]).sum())
print(f"cidade: {u['cidade'].nunique()} grafias -> {u['cidade_norm'].nunique()} cidades")
display(u.groupby("cidade_norm")["cidade"].agg(grafias="nunique", contas="size").sort_values("contas", ascending=False))
achado("Validade", "cidade com grafias diferentes (texto livre)",
       f"{u['cidade'].nunique()} grafias para {u['cidade_norm'].nunique()} cidades (Sampa, SP, S. Paulo, BSB, Floripa...); "
       f"{n_cid:,} fora da grafia oficial", n_cid,
       "Padronizar pela tabela de grafias deste notebook; a uf já vem certa e pode ser usada no lugar da cidade")

achado("Validade", "canal_aquisicao, relogio_conectado, uf e versao_app fora da lista observada",
       "listas sem variação de grafia: 7 canais, 3 relógios, 11 UFs, 6 versões", 0, "Nenhum")

cidade: 36 grafias -> 13 cidades


,grafias,contas
cidade_norm,,
São Paulo,8,27467
Rio de Janeiro,6,7513
Belo Horizonte,5,4550
Curitiba,1,3922
Brasília,5,3268
Porto Alegre,1,3267
Florianópolis,4,2674
Campinas,1,2578
Goiânia,1,1950


### Idade

Limites plausíveis para quem cria conta num app de treino: de 13 a 100 anos no corte. Fora disso, o valor é padrão de
formulário ou erro de digitação.

In [15]:
print(u["idade"].describe().round(1).to_dict())
validas = ~u["data_nascimento"].isin(["", "1900-01-01", "2031-05-12"])
print(f"sem 1900-01-01 e 2031-05-12, a idade vai de {u.loc[validas, 'idade'].min():.1f} a {u.loc[validas, 'idade'].max():.1f} anos")
futuro = u["nasc"] > CORTE
print("nascimentos depois do corte:", u.loc[futuro, "data_nascimento"].value_counts().to_dict())
acima_100 = int((u["idade"] > 100).sum())
achado("Validade", "nascimento no futuro (idade negativa)",
       f"{futuro.sum():,} contas, todas com 2031-05-12", int(futuro.sum()),
       "Tratar como nascimento não informado")
achado("Validade", "idade acima de 100 anos", f"{acima_100:,}: são os 1900-01-01 da Completude (mesma causa)",
       acima_100, "Ver 'nascimento disfarçado de vazio'", status="atenção")

{'count': 61647.0, 'mean': 48.4, 'std': 13.2, 'min': -4.6, '25%': 39.8, '50%': 47.1, '75%': 55.4, 'max': 126.7}
sem 1900-01-01 e 2031-05-12, a idade vai de 18.0 a 74.7 anos
nascimentos depois do corte: {'2031-05-12': 194}


### CEP

`cep_prefixo` são os 3 primeiros dígitos do CEP. Os CEPs de São Paulo capital começam com 0 (01000 a 08499); se o
prefixo foi gravado como número, o zero da frente some.

In [16]:
print("tamanho do cep_prefixo:", u["cep_prefixo"].str.len().value_counts().to_dict())
dois = u["cep_prefixo"].str.len() == 2
print("2 dígitos por uf:", u.loc[dois, "uf"].value_counts().to_dict(), "| por cidade:", u.loc[dois, "cidade_norm"].value_counts().to_dict())
cn = pd.to_numeric(u["cep_prefixo"].replace("", None))
print(f"faixa dos de 2 dígitos: {cn[dois].min():.0f} a {cn[dois].max():.0f}")
sp_cap3 = (u["cidade_norm"] == "São Paulo") & (u["cep_prefixo"].str.len() == 3)
print(f"São Paulo capital com 3 dígitos: {sp_cap3.sum():,}, começando com 0: {u.loc[sp_cap3, 'cep_prefixo'].str.startswith('0').sum():,}")
achado("Validade", "cep_prefixo com 2 dígitos (zero à esquerda perdido)",
       f"{dois.sum():,} contas, todas de São Paulo capital, valores de 10 a 83 (seriam 010 a 083)", int(dois.sum()),
       "Completar com zero à esquerda (zfill(3)) antes de usar o CEP")

tamanho do cep_prefixo: {3: 61960, 0: 1894, 2: 1094}
2 dígitos por uf: {'SP': 1094} | por cidade: {'São Paulo': 1094}


faixa dos de 2 dígitos: 10 a 83
São Paulo capital com 3 dígitos: 25,583, começando com 0: 25,583


## 3. Consistência

Regras entre colunas da mesma linha e comparação com os outros arquivos.

In [17]:
cidade_uf = u.groupby("cidade_norm")["uf"].nunique()
n_cid_uf = int((cidade_uf > 1).sum())
display(u.groupby("cidade_norm")["uf"].agg(lambda s: sorted(s.unique())).to_frame("uf"))
achado("Consistência", "cidade com mais de uma uf", f"{n_cid_uf} cidades", n_cid_uf, "Nenhum")

FAIXA_CEP = {"SP": (10, 199), "RJ": (200, 289), "MG": (300, 399), "BA": (400, 489), "PE": (500, 569),
             "CE": (600, 639), "DF": (700, 736), "GO": (737, 767), "PR": (800, 879), "SC": (880, 899), "RS": (900, 999)}
cep_num = pd.to_numeric(u["cep3"].replace("", None))
lim = u["uf"].map(FAIXA_CEP)
fora_faixa = cep_num.notna() & ~pd.Series([a <= c <= b if pd.notna(c) else True for c, (a, b) in zip(cep_num, lim)], index=u.index)
display(u.assign(cep=cep_num).groupby("cidade_norm")["cep"].agg(["min", "max"]))
achado("Consistência", "CEP fora da faixa da uf (com o zero à esquerda reposto)", f"{fora_faixa.sum():,}",
       int(fora_faixa.sum()), "Nenhum")

,uf
cidade_norm,
Belo Horizonte,[MG]
Brasília,[DF]
Campinas,[SP]
Curitiba,[PR]
Florianópolis,[SC]
Fortaleza,[CE]
Goiânia,[GO]
Porto Alegre,[RS]
Recife,[PE]


,min,max
cidade_norm,,
Belo Horizonte,301.0,313.0
Brasília,700.0,722.0
Campinas,130.0,131.0
Curitiba,800.0,822.0
Florianópolis,880.0,882.0
Fortaleza,600.0,611.0
Goiânia,740.0,749.0
Porto Alegre,900.0,918.0
Recife,500.0,528.0


In [18]:
sist_rel = pd.crosstab(u["sistema_norm"], u["relogio_conectado"])
display(sist_rel)
incoerente = int(((u["relogio_conectado"] == "apple_watch") & (u["sistema_norm"] != "ios")).sum())
achado("Consistência", "Apple Watch em conta que não é iOS", f"{incoerente:,}; e Garmin só aparece em Android",
       incoerente, "Nenhum")

relogio_conectado,apple_watch,garmin,nenhum
sistema_norm,,,
android,0,13403,21748
ios,11386,0,18411


### Relógio da conta contra a origem dos treinos

`relogio_conectado` é o relógio ligado à conta hoje. Uma conta com treino sincronizado do Garmin (ou do Apple Health)
deveria ter esse relógio no cadastro. O contrário não vale: quem tem relógio pode não treinar.

In [19]:
ativ = ler("atividades.parquet", columns=["user_id", "inicio", "origem"])
origens = ativ.groupby(["user_id", "origem"]).size().unstack(fill_value=0)
rel = u.set_index("user_id")["relogio_conectado"]
tem_garmin = origens.index[origens.get("garmin", 0) > 0]
tem_apple = origens.index[origens.get("apple_health", 0) > 0]
cruz = pd.DataFrame({
    "contas com treino do Garmin": rel.loc[tem_garmin].value_counts(),
    "contas com treino do Apple Health": rel.loc[tem_apple].value_counts(),
    "contas no cadastro": rel.value_counts(),
}).fillna(0).astype(int)
display(cruz)
contradiz = int((rel.loc[tem_garmin] != "garmin").sum() + (rel.loc[tem_apple] != "apple_watch").sum())
achado("Consistência", "treino de relógio em conta com outro relógio (ou nenhum) no cadastro", f"{contradiz:,}",
       contradiz, "Nenhum")

,contas com treino do Garmin,contas com treino do Apple Health,contas no cadastro
relogio_conectado,,,
apple_watch,0,6337,11386
garmin,7360,0,13403
nenhum,0,0,40159


### Registro anterior ao cadastro

Nenhuma conta pode ter nascido depois de se cadastrar, nem ter assinatura, evento, cobrança, treino ou amizade antes
do dia do cadastro. A comparação é por dia (a seção 5 mostra por que a hora do cadastro não serve).

In [20]:
nasc_depois = int((u["nasc"] > u["dc"]).sum())
achado("Consistência", "nascimento depois do cadastro", f"{nasc_depois:,}: são os 2031-05-12 da Validade (mesma causa)",
       nasc_depois, "Ver 'nascimento no futuro'", status="atenção")

data_cad = u.set_index("user_id")["data_cad"]
eventos = ler("eventos_assinatura.csv", usecols=["user_id", "data_hora", "evento"])
eventos["data_br"] = para_brasilia(eventos["data_hora"]).dt.normalize()
pagamentos = ler("pagamentos.csv", usecols=["user_id", "data_hora"])
pagamentos["data_br"] = para_brasilia(pagamentos["data_hora"]).dt.normalize()
garmin = ativ["origem"] == "garmin"
ativ["data_br"] = pd.to_datetime(ativ["inicio"]).dt.normalize()
ativ.loc[garmin, "data_br"] = (pd.to_datetime(ativ.loc[garmin, "inicio"]) - pd.Timedelta(hours=3)).dt.normalize()
amizades = ler("amizades.csv")
amizades["d"] = pd.to_datetime(amizades["desde"], format="%Y-%m-%d")

antes_cad = {
    "assinaturas (inicio)": int((assinaturas["ini"] < assinaturas["user_id"].map(data_cad)).sum()),
    "eventos_assinatura": int((eventos["data_br"] < eventos["user_id"].map(data_cad)).sum()),
    "pagamentos": int((pagamentos["data_br"] < pagamentos["user_id"].map(data_cad)).sum()),
    "atividades": int((ativ["data_br"] < ativ["user_id"].map(data_cad)).sum()),
    "amizades (seguidor)": int((amizades["d"] < amizades["seguidor_id"].map(data_cad)).sum()),
    "amizades (seguido)": int((amizades["d"] < amizades["seguido_id"].map(data_cad)).sum()),
}
print(antes_cad)

{'assinaturas (inicio)': 42, 'eventos_assinatura': 0, 'pagamentos': 0, 'atividades': 0, 'amizades (seguidor)': 0, 'amizades (seguido)': 0}


Hipótese para as assinaturas que começam antes do cadastro: são as contas de teste (plano `cortesia`, canal `interno`),
que não passam pelo fluxo normal.

In [21]:
ass_antes = assinaturas[assinaturas["ini"] < assinaturas["user_id"].map(data_cad)]
print("plano:", ass_antes["plano_atual"].value_counts().to_dict(), "| canal:", ass_antes["canal_cobranca"].value_counts().to_dict(),
      "| inicio:", ass_antes["inicio"].value_counts().to_dict())
print("são contas de teste:", ass_antes["user_id"].map(u.set_index("user_id")["eh_teste"]).sum(), "de", len(ass_antes))
n_ass_antes = len(ass_antes)
outros_antes = sum(v for k, v in antes_cad.items() if not k.startswith("assinaturas"))
achado("Consistência", "assinatura que começa antes do cadastro",
       f"{n_ass_antes} assinaturas, todas de contas de teste (cortesia, interno, inicio 2025-01-10)", n_ass_antes,
       "Saem com as contas de teste (decisão já registrada)", status="esperado")
achado("Consistência", "evento, cobrança, treino ou amizade antes do dia do cadastro", f"{outros_antes:,}", outros_antes,
       "Nenhum")

plano: {'cortesia': 42} | canal: {'interno': 42} | inicio: {'2025-01-10': 42}
são contas de teste: 42 de 42


## 4. Unicidade

`user_id` não deve repetir. Como o e-mail identifica a pessoa, o mesmo e-mail em duas contas (ignorando maiúsculas) é
a mesma pessoa com dois cadastros.

In [22]:
achado("Unicidade", "user_id repetido", f"{u['user_id'].duplicated().sum():,}", int(u["user_id"].duplicated().sum()), "Nenhum")
achado("Unicidade", "e-mail repetido exatamente como gravado", f"{u['email'].duplicated().sum():,}",
       int(u["email"].duplicated().sum()), "Nenhum")
rep = u["email_norm"].duplicated(keep=False)
print(f"contas com e-mail repetido ignorando maiúsculas: {rep.sum():,} em {u.loc[rep, 'email_norm'].nunique():,} e-mails")
print("contas por e-mail repetido:", u.loc[rep].groupby("email_norm").size().value_counts().to_dict())

contas com e-mail repetido ignorando maiúsculas: 1,918 em 959 e-mails
contas por e-mail repetido: {2: 959}


Hipótese: em cada par, uma conta é a original e a outra é uma cópia gerada depois, com o e-mail em maiúsculas.
Se for isso, a cópia é sempre a mais nova, tem o perfil idêntico ao da original e nenhum registro em outro arquivo.

In [23]:
par = u.loc[rep].sort_values(["email_norm", "dc"]).copy()
par["ordem"] = par.groupby("email_norm").cumcount().map({0: "mais antiga", 1: "mais nova"})
par["maiuscula"] = par["email"] != par["email_norm"]
display(pd.crosstab(par["ordem"], par["maiuscula"]))

perfil = ["data_nascimento", "sexo", "cidade", "uf", "cep_prefixo", "canal_aquisicao", "sistema", "versao_app", "relogio_conectado"]
a = par[par["ordem"] == "mais antiga"].set_index("email_norm")
b = par[par["ordem"] == "mais nova"].set_index("email_norm").loc[a.index]
iguais = pd.Series({c: round(100 * (a[c] == b[c]).mean(), 1) for c in perfil}, name="% de pares iguais")
display(iguais.to_frame())
dias = (b["dc"] - a["dc"]).dt.days
print(f"dias entre a conta antiga e a nova: de {dias.min()} a {dias.max()} (mediana {dias.median():.0f})")

maiuscula,False,True
ordem,,
mais antiga,959,0
mais nova,0,959


,% de pares iguais
data_nascimento,100.0
sexo,100.0
cidade,100.0
uf,100.0
cep_prefixo,100.0
canal_aquisicao,100.0
sistema,100.0
versao_app,100.0
relogio_conectado,100.0


dias entre a conta antiga e a nova: de 1 a 199 (mediana 98)


In [24]:
copias = set(par.loc[par["maiuscula"], "user_id"])
originais = set(par.loc[~par["maiuscula"], "user_id"])
uso = pd.DataFrame({
    "cópias presentes": {k: len(v & copias) for k, v in ids_por_arquivo.items()},
    "originais presentes": {k: len(v & originais) for k, v in ids_por_arquivo.items()},
})
uso.index.name = "arquivo:coluna (contas distintas)"
display(uso)
u["eh_copia"] = u["user_id"].isin(copias)
n_copias = len(copias)
registros_copias = int(uso["cópias presentes"].sum())
achado("Unicidade", "mesma pessoa com duas contas (e-mail igual sem considerar maiúsculas)",
       f"{n_copias:,} pares; a conta mais nova tem sempre o e-mail em maiúsculas, perfil 100% igual, criada 1 a 199 dias "
       f"depois, e {registros_copias} delas aparece em algum outro arquivo. Hipótese a confirmar com o dono do dado: cópia "
       "gerada pelo sistema", n_copias,
       "Remover a conta com e-mail em maiúsculas (a cópia) e manter a original")

,cópias presentes,originais presentes
arquivo:coluna (contas distintas),,
assinaturas.csv:user_id,0,472
eventos_assinatura.csv:user_id,0,764
pagamentos.csv:user_id,0,461
uso_mensal.csv:user_id,0,958
atividades.parquet:user_id,0,519
amizades.csv:seguidor_id,0,749
amizades.csv:seguido_id,0,751
campanha_ago26.csv:user_id,0,178
onboarding.csv:user_id,0,959


In [25]:
sem_id = u[usuarios.columns.drop("user_id")].duplicated().sum()
achado("Unicidade", "linha inteira repetida sem o user_id", f"{sem_id:,}", int(sem_id), "Nenhum")

## 5. Temporalidade

Espera-se cadastro de 09/2020 (lançamento) até o corte, em horário de Brasília (o dicionário não diz o fuso), com volume
mensal que acompanhe o crescimento de assinantes.

In [26]:
print(f"cadastros de {u['dc'].min()} a {u['dc'].max()}")
achado("Temporalidade", "cadastro antes do lançamento (09/2020)", f"{(u['dc'] < LANCAMENTO).sum():,}",
       int((u["dc"] < LANCAMENTO).sum()), "Nenhum")
pos = u["dc"] >= FIM_CORTE
print("depois do corte, por mês:", u.loc[pos, "mes_cad"].astype(str).value_counts().sort_index().to_dict())
print(f"desses, cópias da seção 4: {(pos & u['eh_copia']).sum()} de {pos.sum()}")
achado("Temporalidade", "cadastro depois do corte (até 04/2027, data no futuro)",
       f"{pos.sum():,} contas, todas cópias da seção 4 (mesma causa); nenhuma tem registro em outro arquivo",
       int(pos.sum()), "Saem com as cópias; nenhuma análise usa cadastro posterior ao corte")

cadastros de 2020-09-01 12:00:00 a 2027-04-04 10:44:09
depois do corte, por mês: {'2026-10': 43, '2026-11': 21, '2026-12': 14, '2027-01': 19, '2027-02': 16, '2027-03': 5, '2027-04': 1}
desses, cópias da seção 4: 119 de 119


### Fuso e hora do cadastro

O dicionário não diz o fuso. Teste: o teste grátis começa no mesmo dia do cadastro? Comparo o dia do cadastro com o dia
do `trial_iniciado` em horário de Brasília e lido como texto (UTC das lojas sem conversão).

In [27]:
trial = eventos[eventos["evento"] == "trial_iniciado"].copy()
trial["dh_br"] = para_brasilia(trial["data_hora"])
trial["dia_texto"] = pd.to_datetime(trial["data_hora"].str[:10])
primeiro = trial.sort_values("dh_br").drop_duplicates("user_id").set_index("user_id")
dc = u.set_index("user_id")["dc"].loc[primeiro.index]
dif_br = (primeiro["dh_br"].dt.normalize() - dc.dt.normalize()).dt.days
dif_txt = (primeiro["dia_texto"] - dc.dt.normalize()).dt.days
print("dias entre trial e cadastro, em Brasília:", dif_br.value_counts().to_dict())
print("dias entre trial e cadastro, lendo o texto:", dif_txt.value_counts().to_dict())
achado("Temporalidade", "fuso de data_cadastro (dicionário não diz)",
       f"em horário de Brasília, o trial cai no dia do cadastro em {(dif_br == 0).sum():,} de {len(dif_br):,} contas; "
       f"lendo o trial como texto, {(dif_txt != 0).sum():,} cairiam em outro dia: o cadastro está em horário de Brasília",
       0, "Usar data_cadastro como horário de Brasília")

dias entre trial e cadastro, em Brasília: {0: 46996}
dias entre trial e cadastro, lendo o texto: {0: 43273, 1: 3723}


Se a hora também fosse real, o trial viria minutos depois do cadastro. A diferença em horas e o horário fixo 12:00:00
dizem se a hora serve.

In [28]:
horas = (primeiro["dh_br"] - dc).dt.total_seconds() / 3600
print(f"trial - cadastro, em horas: de {horas.min():.1f} a {horas.max():.1f}; trial antes do cadastro: {(horas < 0).sum():,} de {len(horas):,}")
meio_dia = u["data_cadastro"].str.endswith(" 12:00:00")
print(f"cadastro às 12:00:00: {meio_dia.sum():,}; antes de 07/2024: {(meio_dia & (u['dc'] < INICIO_LOG)).sum():,} de {(u['dc'] < INICIO_LOG).sum():,}")
print("hora do cadastro (07/2024 em diante):", u.loc[u["dc"] >= INICIO_LOG, "dc"].dt.hour.value_counts().sort_index().to_dict())
achado("Temporalidade", "hora do cadastro não confiável",
       f"antes de 07/2024 só a data foi gravada ({(meio_dia & (u['dc'] < INICIO_LOG)).sum():,} contas às 12:00:00); "
       f"depois, o trial começa até {abs(horas.min()):.0f}h antes ou {horas.max():.0f}h depois do cadastro, mas sempre no mesmo dia",
       N, "Usar só a data do cadastro; no mesmo dia, considerar o cadastro anterior a qualquer evento")

trial - cadastro, em horas: de -16.9 a 16.8; trial antes do cadastro: 23,588 de 46,996
cadastro às 12:00:00: 5,283; antes de 07/2024: 5,275 de 5,275
hora do cadastro (07/2024 em diante): {6: 3433, 7: 3357, 8: 3452, 9: 3590, 10: 3507, 11: 3560, 12: 3561, 13: 3600, 14: 3585, 15: 3407, 16: 3451, 17: 3568, 18: 3468, 19: 3544, 20: 3507, 21: 3512, 22: 3571}


### Volume por mês

Salto acima de 50% só é falha se os assinantes não acompanharem. Comparo com as assinaturas iniciadas no mês.

In [29]:
mensal = pd.DataFrame({
    "cadastros": u.loc[u["dc"] < FIM_CORTE].groupby("mes_cad").size(),
    "assinaturas iniciadas": assinaturas.groupby(assinaturas["ini"].dt.to_period("M")).size(),
}).fillna(0).astype(int)
meses = pd.period_range("2020-09", "2026-09", freq="M")
faltando = meses.difference(mensal.index)
mensal = mensal.reindex(meses, fill_value=0)
mensal["var cadastros %"] = (100 * (mensal["cadastros"] / mensal["cadastros"].shift() - 1)).round(0)
mensal["var assinaturas %"] = (100 * (mensal["assinaturas iniciadas"] / mensal["assinaturas iniciadas"].shift() - 1)).round(0)
mensal["cadastros por assinatura"] = (mensal["cadastros"] / mensal["assinaturas iniciadas"]).round(2)
display(mensal.loc["2024-01":"2026-09"])
print("cadastros por assinatura, antes e depois de 07/2024:",
      mensal.loc[:"2024-06", "cadastros"].sum() / mensal.loc[:"2024-06", "assinaturas iniciadas"].sum(),
      mensal.loc["2024-07":, "cadastros"].sum() / mensal.loc["2024-07":, "assinaturas iniciadas"].sum())
saltos = mensal[mensal["var cadastros %"].abs() > 50]
display(saltos)
achado("Temporalidade", "mês sem cadastro", f"{len(faltando)}", len(faltando), "Nenhum")

,cadastros,assinaturas iniciadas,var cadastros %,var assinaturas %,cadastros por assinatura
2024-01,120,115,4.0,-7.0,1.04
2024-02,116,116,-3.0,1.0,1.00
2024-03,116,115,0.0,-1.0,1.01
2024-04,127,123,9.0,7.0,1.03
2024-05,118,108,-7.0,-12.0,1.09
2024-06,79,113,-33.0,5.0,0.70
2024-07,1763,644,2132.0,470.0,2.74
2024-08,1947,918,10.0,43.0,2.12
2024-09,2064,927,6.0,1.0,2.23
2024-10,1997,973,-3.0,5.0,2.05


cadastros por assinatura, antes e depois de 07/2024: 1.0144230769230769 2.1700980213533505


,cadastros,assinaturas iniciadas,var cadastros %,var assinaturas %,cadastros por assinatura
2024-07,1763,644,2132.0,470.0,2.74
2025-01,3282,1483,105.0,96.0,2.21
2026-01,3513,1561,99.0,91.0,2.25


In [30]:
jan = saltos.index[saltos.index.month == 1]
n_jan = int(mensal.loc[jan, "cadastros"].sum())
achado("Temporalidade", "salto acima de 50% nos janeiros",
       f"{', '.join(str(m) for m in jan)}: cadastros +{saltos.loc[jan, 'var cadastros %'].min():.0f}% ou mais, e as "
       f"assinaturas sobem junto (+{saltos.loc[jan, 'var assinaturas %'].min():.0f}% ou mais): sazonalidade de começo de ano",
       n_jan, "Nenhum; considerar o mês de cadastro nas análises", status="esperado")
achado("Temporalidade", "salto de 07/2024 (mudança de universo)",
       f"de {int(mensal.loc['2024-06', 'cadastros'])} para {int(mensal.loc['2024-07', 'cadastros']):,} cadastros; a razão "
       "cadastros/assinaturas passa de ~1 para ~2: é a ausência das contas grátis antigas (Completude, mesma causa)",
       int(mensal.loc["2024-07", "cadastros"]), "Ver 'contas grátis cadastradas antes de 07/2024 ausentes'", status="atenção")

### Colunas sem data

`versao_app` é a "última versão do app vista", e `relogio_conectado` e `sistema` descrevem a conta hoje. O arquivo não
diz quando foram observadas: podem ter mudado depois do corte (ou depois que a pessoa saiu). Num modelo de saída, é
risco de vazamento.

In [31]:
display(pd.crosstab(u["versao_app"].rename("versao_app"), u["assina"] & u["user_id"].isin(
    assinaturas.loc[assinaturas["status_atual"] == "ativa", "user_id"]).rename("situação")).rename(columns={True: "ativa no retrato", False: "outras"}))
achado("Temporalidade", "colunas sem data de observação (versao_app, relogio_conectado, sistema)",
       "valores do dia da exportação, sem data; podem refletir o que aconteceu depois do corte", N,
       "Não usar como preditor sem data; para relógio, usar a origem dos treinos do parquet até o corte")

col_0,outras,ativa no retrato
versao_app,,
4.9.8,2010,594
5.1.0,2063,581
5.3.0,7460,2335
5.3.2,7795,2104
5.4.0,15378,4618
5.4.1,15429,4581


## 6. Resumo

In [32]:
resumo = pd.DataFrame(achados)
resumo["% do arquivo"] = (100 * resumo["linhas_afetadas"] / N).round(2)
resumo.to_csv(SAIDAS / "qualidade_usuarios.csv", index=False)
print(resumo["status"].value_counts().to_dict())
resumo

{'ok': 17, 'atenção': 15, 'esperado': 5}


,criterio,checagem,status,resultado,linhas_afetadas,tratamento_sugerido,% do arquivo
0,Completude,vazio nas colunas preenchidas pelo app ou pelo...,ok,0,0,Nenhum,0.00
1,Completude,nascimento vazio,esperado,"3,301 (5.1%); taxa por ano de 4.8% a 6.2%",3301,Campo digitado e opcional na prática: manter c...,5.08
2,Completude,sexo vazio,esperado,"3,142 (4.8%); taxa por ano de 4.1% a 4.9%",3142,Campo digitado e opcional na prática: manter c...,4.84
3,Completude,CEP vazio,esperado,"1,894 (2.9%); taxa por ano de 2.8% a 3.3%",1894,Campo digitado e opcional na prática: manter c...,2.92
4,Completude,nascimento disfarçado de vazio (1900-01-01),atenção,412 contas com 1900-01-01 (idade de 126 anos);...,412,Tratar como nascimento não informado (somar ao...,0.63
5,Completude,contas de teste (@vitaliza.com.br),atenção,"60 contas, as mesmas 60 do canal 'interno'",60,Já registrado no decisoes.md: remover,0.09
6,Completude,user_id dos outros 11 arquivos sem cadastro aqui,ok,0 linhas,0,Nenhum,0.00
7,Completude,contas grátis cadastradas antes de 07/2024 aus...,atenção,"5,275 contas antes de 07/2024, 98.7% viraram a...",5275,Não usar coortes de cadastro anteriores a 07/2...,8.12
8,Validade,user_id fora do padrão u_[0-9a-f]{8},ok,0,0,Nenhum,0.00
9,Validade,e-mail com formato inválido,ok,0,0,Nenhum,0.00


In [33]:
hash_fim = {f: sha1(BRUTOS / f) for f in ARQUIVOS}
assert hash_fim == hash_inicio
print("dados/brutos/ inalterado:", hash_fim == hash_inicio)

dados/brutos/ inalterado: True
